# Bootcamp Playbook

Welcome to the bootcamp! This three-part notebook series introduces NVIDIA Nemotron reasoning controls, shows how to build a movie database MCP server, and connects that server to the NVIDIA NeMo Agent Toolkit.

## Table of Contents

1. [NVIDIA Nemotron Reasoning Controls](01_reasoning_controls_notebook.ipynb)
2. [Movie Database MCP Server](02_movie_database_mcp.ipynb)
3. [NeMo Agent Toolkit with the Movie MCP Server](03_nemo_agent_toolkit.ipynb)


# NVIDIA Nemotron Reasoning Controls

This self-contained notebook focuses on three controls from the Nemotron 3 Super getting-started example:

1. `enable_thinking: true`
2. `reasoning_budget`
3. `low_effort: true`

The goal is to give workshop participants a compact, hands-on way to compare deeper reasoning, bounded reasoning, and faster low-effort reasoning on the same prompts.

## 1. Configure the API Key, Endpoint, and Model

The notebook defaults to Nemotron 3 Super because that is the model used in the referenced getting-started guide. You can override the model by setting `NEMOTRON_MODEL` before running the notebook.

In [ ]:
import os
from getpass import getpass

from openai import OpenAI


if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("Enter your NVIDIA API Key: ").strip()

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1/"
MODEL = os.environ.get("NEMOTRON_MODEL", "nvidia/nemotron-3-super-120b-a12b")

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=os.environ["NVIDIA_API_KEY"],
    default_headers={"NVCF-POLL-SECONDS": "1800"},
)

print(f"Configured endpoint: {NVIDIA_BASE_URL}")
print(f"Configured model: {MODEL}")

## 2. Create Reusable Streaming Helpers

Reasoning-capable Nemotron responses can stream reasoning content separately from the final answer. The helper below checks both `reasoning_content` and `reasoning`, then prints the reasoning in gray and the final answer in the default color.

In [ ]:
import json
import time
from typing import Any


GRAY = "\033[90m"
RESET = "\033[0m"


def make_reasoning_extra_body(
    *,
    enable_thinking: bool = True,
    reasoning_budget: int | None = None,
    low_effort: bool | None = None,
) -> dict[str, Any]:
    """Build the extra_body payload for Nemotron reasoning controls."""
    chat_template_kwargs = {"enable_thinking": enable_thinking}

    if low_effort is not None:
        chat_template_kwargs["low_effort"] = low_effort

    extra_body: dict[str, Any] = {"chat_template_kwargs": chat_template_kwargs}

    if reasoning_budget is not None:
        extra_body["reasoning_budget"] = reasoning_budget

    return extra_body


def preview_payload(extra_body: dict[str, Any]) -> None:
    """Print only the reasoning-control portion of the request."""
    print(json.dumps(extra_body, indent=2))


def stream_with_reasoning(completion, *, show_reasoning: bool = True) -> dict[str, Any]:
    """Stream a response, separate reasoning from the final answer, and return both."""
    reasoning = ""
    answer = ""
    started_at = time.perf_counter()
    in_reasoning = False

    for chunk in completion:
        if not getattr(chunk, "choices", None):
            continue

        delta = chunk.choices[0].delta
        reasoning_piece = (
            getattr(delta, "reasoning_content", None)
            or getattr(delta, "reasoning", None)
        )
        content_piece = getattr(delta, "content", None)

        if reasoning_piece:
            reasoning += reasoning_piece
            if show_reasoning:
                if not in_reasoning:
                    print(GRAY, end="")
                    in_reasoning = True
                print(reasoning_piece, end="", flush=True)

        if content_piece:
            answer += content_piece
            if in_reasoning:
                print(RESET, end="")
                in_reasoning = False
            print(content_piece, end="", flush=True)

    if in_reasoning:
        print(RESET, end="")

    print()
    elapsed_seconds = time.perf_counter() - started_at
    return {
        "reasoning": reasoning,
        "answer": answer,
        "reasoning_chars": len(reasoning),
        "answer_chars": len(answer),
        "elapsed_seconds": elapsed_seconds,
    }


def run_demo(
    prompt: str,
    *,
    extra_body: dict[str, Any],
    system_prompt: str = "You are a helpful NVIDIA Nemotron assistant.",
    max_tokens: int = 4096,
    temperature: float = 1.0,
    top_p: float = 0.95,
    show_reasoning: bool = True,
) -> dict[str, Any]:
    """Create a streamed chat completion and display the reasoning and answer."""
    print("Request reasoning controls:")
    preview_payload(extra_body)
    print("\nStreamed response:\n")

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        extra_body=extra_body,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        stream=True,
        timeout=1800,
    )

    result = stream_with_reasoning(completion, show_reasoning=show_reasoning)
    print(
        "\nSummary: "
        f"reasoning_chars={result['reasoning_chars']:,}, "
        f"answer_chars={result['answer_chars']:,}, "
        f"elapsed_seconds={result['elapsed_seconds']:.1f}"
    )
    return result

## 3. Baseline: Thinking Off

This is the fast, direct control run. Run it first so participants can compare how the same prompt behaves when reasoning is enabled.

In [ ]:
comparison_prompt = """
A workshop has three demo stations: inference, reasoning, and agents.
Each station takes 12 minutes. Participants need 3 minutes to move between stations.
Can a group complete all three stations in 45 minutes? Explain briefly.
"""

baseline_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=False),
    max_tokens=1024,
    temperature=0,
    top_p=1,
    show_reasoning=False,
)

## 4. `enable_thinking: true`

Turning thinking on asks the model to reason before producing the final user-facing response. In the stream, the reasoning appears separately from the answer.

In [ ]:
thinking_result = run_demo(
    comparison_prompt,
    extra_body=make_reasoning_extra_body(enable_thinking=True),
    max_tokens=2048,
    temperature=1.0,
    top_p=0.95,
    show_reasoning=True,
)

## 5. `reasoning_budget`

`reasoning_budget` limits how much reasoning the model can use. Lower budgets are useful for latency-sensitive tasks, while larger budgets are useful for tasks that require more multi-step planning.

In [ ]:
budget_prompt = """
Create a 45-minute hands-on mini-agenda for ML engineers learning Nemotron.
Constraints:
- Include one API warmup, one reasoning-control demo, and one agentic workflow demo.
- Leave 5 minutes for Q&A.
- Keep transitions realistic.
- Return a minute-by-minute agenda and explain the tradeoffs.
"""

small_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=1024,
    ),
    max_tokens=3072,
    show_reasoning=True,
)

In [ ]:
larger_budget_result = run_demo(
    budget_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=8192,
    ),
    max_tokens=8192,
    show_reasoning=True,
)

## 6. `low_effort: true`

`low_effort` keeps thinking enabled but asks for a shorter, faster reasoning path. It is a good fit when you want a reasoning-capable mode without deep exploration.

In [ ]:
low_effort_prompt = """
A participant asks: should I use reasoning mode for every chatbot request?
Give a practical answer with examples of when to use thinking, bounded thinking,
low-effort thinking, and thinking off.
"""

low_effort_result = run_demo(
    low_effort_prompt,
    extra_body=make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    max_tokens=2048,
    show_reasoning=True,
)

## 7. Compare the Runs

The exact numbers vary by run, but this table makes the tradeoffs visible: reasoning length, answer length, and elapsed time.

In [ ]:
results = [
    ("thinking_off", baseline_result),
    ("thinking_on", thinking_result),
    ("budget_1024", small_budget_result),
    ("budget_8192", larger_budget_result),
    ("low_effort", low_effort_result),
]

header = f"{'run':<20} {'reasoning_chars':>16} {'answer_chars':>14} {'elapsed_seconds':>16}"
print(header)
print("-" * len(header))

for name, result in results:
    print(
        f"{name:<20} "
        f"{result['reasoning_chars']:>16,} "
        f"{result['answer_chars']:>14,} "
        f"{result['elapsed_seconds']:>16.1f}"
    )

## 8. Workshop Recipes

Use these request shapes as a quick reference during the demo.

In [ ]:
recipes = {
    "direct_answer": make_reasoning_extra_body(enable_thinking=False),
    "thinking_on": make_reasoning_extra_body(enable_thinking=True),
    "bounded_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=4096,
    ),
    "low_effort_thinking": make_reasoning_extra_body(
        enable_thinking=True,
        low_effort=True,
    ),
    "low_effort_with_budget": make_reasoning_extra_body(
        enable_thinking=True,
        reasoning_budget=2048,
        low_effort=True,
    ),
}

for name, payload in recipes.items():
    print(f"\n{name}")
    print(json.dumps(payload, indent=2))

## 9. Participant Exercise

Change the prompt and recipe below. Before running it, ask participants to predict which mode will offer the best trade-off between latency and quality.

In [ ]:
my_prompt = """
Design a two-slide explanation of reasoning_budget for an engineering audience.
Slide 1 should explain the control. Slide 2 should explain when to tune it.
"""

my_recipe = recipes["low_effort_with_budget"]

my_result = run_demo(
    my_prompt,
    extra_body=my_recipe,
    max_tokens=3072,
    show_reasoning=True,
)

## Takeaways

- Use `enable_thinking: true` for complex reasoning, planning, logic, and technical problem-solving.
- Use `reasoning_budget` to bound reasoning work and keep latency and cost predictable.
- Use `low_effort: true` when you want reasoning mode with shorter, cheaper responses.
- Use `enable_thinking: false` for simple responses where direct output matters more than reasoning depth.

For most production-facing apps, keep raw reasoning out of the user interface and render only the final answer unless the demo or debugging workflow explicitly needs to inspect the reasoning stream.

## Links and Resources

- **[NVIDIA Nemotron 3 Super Model Card](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)**: Model details and a hosted API example for `nvidia/nemotron-3-super-120b-a12b`.
- **[NVIDIA NIM Thinking Budget Control](https://docs.nvidia.com/nim/large-language-models/1.15.0/thinking-budget-control.html)**: Guidance for configuring `reasoning_budget`, `low_effort`, and model-specific reasoning controls.
- **[NVIDIA RAG Blueprint Reasoning Controls](https://docs.nvidia.com/rag/latest/enable-nemotron-thinking.html)**: An overview of Nemotron reasoning modes and their latency and quality trade-offs.
- **[OpenAI Python Streaming Helpers](https://github.com/openai/openai-python/blob/main/helpers.md)**: Streaming patterns for the OpenAI-compatible Python client used in this notebook.

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.